# CNN & Transfer Learning di CIFAR-10

**Modul 2: Deep Learning Fundamentals** | Notebook 3 dari 6

**Tujuan Pembelajaran (Learning Objectives):**
1. Memahami cara kerja CNN (Convolutional Neural Network)
2. Membangun CNN dari nol untuk klasifikasi gambar 10 kelas
3. Memvisualisasikan apa yang 'dipelajari' oleh CNN
4. Menerapkan Transfer Learning dengan ResNet50
5. Membandingkan CNN dari nol vs Transfer Learning di data validation, lalu sekali di data test

**Estimasi sesi:** ± 50 menit di T4 (termasuk membaca)

**Dataset:** CIFAR-10 — 60.000 gambar berwarna 32×32 pixel dalam 10 kategori (sudah tersedia di TensorFlow, tidak perlu download manual)

In [ ]:
import os
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

tf.random.set_seed(42)
np.random.seed(42)
sns.set_style('whitegrid')

QUICK = bool(os.environ.get('QUICK'))
N_TRAIN = 3000 if QUICK else 45000
N_VAL = 300 if QUICK else 5000
N_TEST = 300 if QUICK else None
N_TRAIN_TRANSFER = 1000 if QUICK else N_TRAIN
EPOCHS_CNN = 1 if QUICK else 15
EPOCHS_TRANSFER = 1 if QUICK else 10

print(f"TensorFlow version: {tf.__version__}")
if tf.config.list_physical_devices('GPU'):
    print("GPU terdeteksi. Untuk CNN dengan batch sebesar ini, training di GPU biasanya jauh lebih cepat; besarnya kita ukur di Notebook 5.")
else:
    print("Tidak ada GPU. Disarankan aktifkan GPU: Runtime -> Change runtime type -> T4 GPU.")

## Bagian 1: Apa itu CNN (Convolutional Neural Network)?

### Analogi: Cara Mata Kita Melihat

Ketika kamu melihat foto kucing, mata kamu tidak melihat SELURUH gambar sekaligus. Kamu perhatikan bagian-bagian kecil: *'Ada telinga runcing... ada kumis... ada mata bulat...'* lalu otak menyimpulkan: **'Ini kucing!'**

CNN bekerja persis seperti itu:
1. **Convolution** = Memindai gambar bagian per bagian (seperti mata scanning)
2. **Pooling** = Menyederhanakan informasi (fokus pada feature penting)
3. **Dense/Flatten** = Menyimpulkan dari feature yang ditemukan

### Perbedaan CNN vs Neural Network Biasa

| Aspek | Neural Network Biasa | CNN |
|-------|---------------------|-----|
| Input | Angka satu baris (1D) | Gambar utuh (2D/3D) |
| Cara kerja | Semua pixel terhubung ke semua neuron | Filter kecil scanning gambar |
| Parameter | Sangat banyak (784 × 128 = 100K+) | Jauh lebih sedikit (filter 3×3 = 9) |
| Cocok untuk | Data tabular | Gambar, video, sinyal |

### Komponen Utama CNN

**1. Convolutional Layer** — Filter kecil (3×3 pixel) bergeser di seluruh gambar, mencari pola lokal (garis, tekstur, bentuk)

**2. Pooling Layer** — Menyusutkan gambar (misal 4×4 → 2×2) dengan mengambil nilai terbesar (Max Pooling). Mengurangi ukuran tapi mempertahankan feature penting.

**3. Flatten + Dense** — Mengubah feature map 2D menjadi 1D, lalu klasifikasi seperti neural network biasa.

In [ ]:
# Demo sederhana: Bagaimana Convolution bekerja
from scipy.signal import convolve2d

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Input "gambar" sederhana (8x8)
np.random.seed(42)
simple_image = np.zeros((8, 8))
simple_image[2:6, 2:6] = 1  # kotak putih di tengah
axes[0].imshow(simple_image, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Input Gambar (8x8)', fontweight='bold')
axes[0].grid(True, alpha=0.3)
for i in range(8):
    for j in range(8):
        axes[0].text(j, i, f'{simple_image[i,j]:.0f}', ha='center', va='center',
                    fontsize=8, color='red' if simple_image[i,j] > 0 else 'gray')

# 2. Filter/Kernel (3x3)
kernel = np.array([[-1, -1, -1],
                   [-1,  8, -1],
                   [-1, -1, -1]])
axes[1].imshow(kernel, cmap='RdBu', vmin=-2, vmax=8)
axes[1].set_title('Filter Edge Detection (3x3)', fontweight='bold')
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f'{kernel[i,j]}', ha='center', va='center',
                    fontsize=12, fontweight='bold')

# 3. Hasil konvolusi
result = convolve2d(simple_image, kernel, mode='valid')
axes[2].imshow(result, cmap='hot')
axes[2].set_title('Hasil Konvolusi (Edge Detected!)', fontweight='bold')

plt.suptitle('Cara Kerja Convolution: Filter Scanning Gambar', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Filter 'bergeser' di gambar, menghitung perkalian elemen per elemen.")
print("Hasilnya = Feature Map yang mendeteksi pola tertentu (misal: tepi/edge).")

## Bagian 2: Dataset CIFAR-10

CIFAR-10 adalah dataset 60.000 gambar berwarna (32×32 pixel) dalam 10 kategori. Dataset ini sudah tersedia di TensorFlow — tidak perlu download manual!

**10 Kategori:** Pesawat, Mobil, Burung, Kucing, Rusa, Anjing, Katak, Kuda, Kapal, Truk

Catatan: saat dijalankan lokal dengan `QUICK=1` (tanpa GPU), notebook ini memakai subset kecil dari data supaya tetap selesai dalam hitungan menit.

In [ ]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['Pesawat', 'Mobil', 'Burung', 'Kucing', 'Rusa',
               'Anjing', 'Katak', 'Kuda', 'Kapal', 'Truk']

if QUICK:
    X_train, y_train = X_train[:N_TRAIN + N_VAL], y_train[:N_TRAIN + N_VAL]
    X_test, y_test = X_test[:N_TEST], y_test[:N_TEST]

print(f"Data training : {X_train.shape[0]:,} gambar")
print(f"Data testing  : {X_test.shape[0]:,} gambar")
print(f"Ukuran gambar : {X_train.shape[1]}×{X_train.shape[2]} pixel, {X_train.shape[3]} channel (RGB)")
print(f"Jumlah kelas  : {len(class_names)}")

In [ ]:
plt.figure(figsize=(12, 6))
for i in range(20):
    plt.subplot(2, 10, i + 1)
    plt.imshow(X_train[i])
    plt.title(class_names[y_train[i][0]], fontsize=8)
    plt.axis('off')
plt.suptitle('Contoh Gambar CIFAR-10 (32×32 pixel)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Gambar ini sangat kecil (32×32) — bahkan manusia kadang sulit membedakan!")

### Normalisasi & Persiapan Data

Seperti di notebook sebelumnya, kita normalisasi pixel dari 0-255 ke 0-1.

In [ ]:
# Normalisasi
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Split train/validation
X_train_final, X_val = X_train[:N_TRAIN], X_train[N_TRAIN:N_TRAIN + N_VAL]
y_train_final, y_val = y_train[:N_TRAIN], y_train[N_TRAIN:N_TRAIN + N_VAL]

print(f"Training : {X_train_final.shape[0]:,} gambar")
print(f"Validation : {X_val.shape[0]:,} gambar")
print(f"Testing  : {X_test.shape[0]:,} gambar")
print(f"Nilai pixel: {X_train_final.min():.1f} - {X_train_final.max():.1f}")

### Baseline — Pembanding Sebelum Model

Sebelum menilai CNN, kita cek dulu "jawaban malas": selalu jawab kelas yang paling sering muncul di data training. Angka ini jadi pembanding minimal yang harus dilewati kedua model nanti.

In [ ]:
from collections import Counter

majority_class = Counter(y_train_final.flatten().tolist()).most_common(1)[0][0]
baseline_acc = (y_test.flatten() == majority_class).mean()

print(f"Kelas mayoritas di data training: {class_names[majority_class]}")
print(f"Baseline (selalu tebak '{class_names[majority_class]}'): akurasi {baseline_acc:.1%}")

## Bagian 3: Membangun CNN dari Nol

Sekarang kita bangun CNN step by step. Arsitekturnya:

```
Input (32×32×3 RGB)
   ↓
Conv2D(32 filter, 3×3) + ReLU → mendeteksi feature dasar (garis, warna)
   ↓
MaxPooling(2×2) → menyusutkan 32×32 → 16×16
   ↓
Conv2D(64 filter, 3×3) + ReLU → mendeteksi feature kompleks (bentuk, tekstur)
   ↓
MaxPooling(2×2) → menyusutkan 16×16 → 8×8
   ↓
Conv2D(128 filter, 3×3) + ReLU → mendeteksi feature abstrak (bagian objek)
   ↓
MaxPooling(2×2) → menyusutkan 8×8 → 4×4
   ↓
Flatten → 4×4×128 = 2,048 angka
   ↓
Dense(128) + ReLU + Dropout(0.5)
   ↓
Dense(10) + Softmax → prediksi 10 kelas
```

In [ ]:
# Augmentasi data sebagai layer Keras — otomatis aktif hanya saat training
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.04),
    tf.keras.layers.RandomZoom(0.1),
], name='augmentation')

model_cnn = tf.keras.Sequential([
    tf.keras.Input(shape=(32, 32, 3)),
    data_augmentation,

    # Block 1: Deteksi feature dasar
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same', name='conv1'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2), name='pool1'),

    # Block 2: Deteksi feature menengah
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv2'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2), name='pool2'),

    # Block 3: Deteksi feature kompleks
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv3'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2), name='pool3'),

    # Classifier
    tf.keras.layers.Flatten(name='flatten'),
    tf.keras.layers.Dense(128, activation='relu', name='dense1'),
    tf.keras.layers.Dropout(0.5, name='dropout'),
    tf.keras.layers.Dense(10, activation='softmax', name='output')
], name='CNN_CIFAR10')

model_cnn.summary()

### Visualisasi Arsitektur CNN

In [ ]:
tf.keras.utils.plot_model(
    model_cnn,
    show_shapes=True,
    show_layer_names=True,
    show_layer_activations=True,
    to_file='cnn_architecture.png',
    dpi=100
)

Dimensi berubah seperti berikut:
- Input: 32×32×3 (3.072 angka)
- Setelah Conv+Pool: ukuran mengecil, tapi jumlah filter bertambah
- Output: 10 angka (probabilitas untuk setiap kelas)

### Compile dan Training

Augmentasi (`RandomFlip`, `RandomRotation`, `RandomZoom`) sudah dipasang sebagai layer di dalam `model_cnn`, jadi otomatis aktif hanya saat `model.fit()` dan otomatis nonaktif saat evaluasi/prediksi — tidak perlu generator Python terpisah (yang berjalan di CPU dan bisa jadi bottleneck saat GPU menganggur menunggu data).

± 6-10 menit di T4 untuk 15 epoch pada 45.000 gambar (mode `QUICK=1` di CPU lokal: ±1-2 menit untuk 1 epoch pada 3.000 gambar).

In [ ]:
model_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Mulai training CNN...")
history_cnn = model_cnn.fit(
    X_train_final, y_train_final,
    batch_size=64,
    epochs=EPOCHS_CNN,
    validation_data=(X_val, y_val),
    verbose=1
)
print("\nTraining selesai.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_cnn.history['loss'], label='Training', linewidth=2)
ax1.plot(history_cnn.history['val_loss'], label='Validation', linewidth=2)
ax1.set_title('Loss (Kesalahan)', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history_cnn.history['accuracy'], label='Training', linewidth=2)
ax2.plot(history_cnn.history['val_accuracy'], label='Validation', linewidth=2)
ax2.set_title('Accuracy (Ketepatan)', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Proses Training CNN', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

val_acc_cnn = history_cnn.history['val_accuracy'][-1]
print(f"\nAkurasi CNN pada data validation (epoch terakhir): {val_acc_cnn:.1%}")

## Bagian 4: Apa yang Dipelajari CNN?

Berikut apa yang 'dilihat' oleh CNN saat memproses gambar.

In [ ]:
# Prediksi beberapa gambar test (ilustrasi — bukan evaluasi akhir)
predictions = model_cnn.predict(X_test[:12], verbose=0)

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for i in range(12):
    ax = axes[i // 6, i % 6]
    ax.imshow(X_test[i])
    pred_class = np.argmax(predictions[i])
    true_class = y_test[i][0]
    color = 'green' if pred_class == true_class else 'red'
    confidence = predictions[i][pred_class]
    ax.set_title(f'{class_names[pred_class]}\n({confidence:.0%})',
                 color=color, fontsize=9)
    ax.axis('off')
plt.suptitle('Prediksi CNN (Hijau=Benar, Merah=Salah)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Visualisasi Feature Maps

Kita bisa melihat APA yang dilihat setiap layer CNN saat memproses gambar.

In [ ]:
# Visualisasi Feature Maps — cara yang kompatibel dengan semua versi TF
sample_img = X_test[0:1]

# Buat model extractor menggunakan Input baru + layer conv1 yang sudah terlatih
extractor_input = tf.keras.Input(shape=(32, 32, 3))
conv1_output = model_cnn.get_layer('conv1')(extractor_input)
feature_model = tf.keras.Model(inputs=extractor_input, outputs=conv1_output)

features = feature_model.predict(sample_img, verbose=0)
print(f"Feature maps shape: {features.shape}")
print(f"  → {features.shape[-1]} filter, masing-masing {features.shape[1]}x{features.shape[2]} pixel\n")

# Tampilkan feature maps
fig, axes = plt.subplots(4, 8, figsize=(14, 7))
fig.suptitle(f'Feature Maps Layer 1 — "{class_names[y_test[0][0]]}"',
             fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < features.shape[-1]:
        ax.imshow(features[0, :, :, i], cmap='viridis')
    ax.axis('off')

plt.tight_layout()
plt.show()

print("Setiap 'kotak' adalah satu filter yang mendeteksi pola berbeda:")
print("Ada yang mendeteksi garis horizontal, vertikal, diagonal, tekstur, dll.")

## Bagian 5: Transfer Learning — Belajar dari Model yang Sudah Pintar

### Analogi: Belajar Bahasa Baru

Bayangkan kamu sudah fasih Bahasa Indonesia, lalu ingin belajar Bahasa Melayu. Kamu tidak mulai dari nol — banyak kata dan grammar yang sama! Kamu 'transfer' pengetahuan Bahasa Indonesia ke Bahasa Melayu.

**Transfer Learning** bekerja sama:
1. Ambil model yang sudah dilatih pada dataset besar (ImageNet: 14 juta gambar, 1000 kelas)
2. 'Bekukan' layer-layer awal (yang sudah pandai mendeteksi feature umum)
3. Ganti layer terakhir sesuai tugas kita
4. Latih ulang hanya layer akhir — jauh lebih sedikit bobot yang perlu dilatih

Model yang kita gunakan: **ResNet50** (50 layer, juara ImageNet 2015)

In [ ]:
# Load ResNet50 tanpa layer klasifikasi teratas
# weights='imagenet' tetap dipakai (unduhan ~100 MB, sekali saja) — termasuk saat QUICK=1
base_model = tf.keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(32, 32, 3)
)

# Bekukan semua layer ResNet (tidak dilatih ulang)
base_model.trainable = False

# Tambahkan layer klasifikasi baru
model_transfer = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
], name='ResNet50_Transfer')

print(f"Total parameter    : {model_transfer.count_params():,}")
print(f"Trainable parameter: {sum(tf.keras.backend.count_params(w) for w in model_transfer.trainable_weights):,}")
print(f"Frozen parameter   : {sum(tf.keras.backend.count_params(w) for w in model_transfer.non_trainable_weights):,}")
print(f"\nHanya {sum(tf.keras.backend.count_params(w) for w in model_transfer.trainable_weights)/model_transfer.count_params():.1%} parameter yang dilatih!")

### Compile dan Training Transfer Learning

Hanya layer classifier baru yang dilatih (backbone ResNet50 dibekukan), tapi forward pass tetap melewati 50 layer ResNet50 — jadi tetap butuh waktu meski jumlah parameter yang dilatih sedikit.

± 6-10 menit di T4 untuk 10 epoch pada 45.000 gambar (mode `QUICK=1` di CPU lokal: ±1-2 menit untuk 1 epoch pada 1.000 gambar — subset lebih kecil dari CNN karena forward pass ResNet50 lebih berat).

In [ ]:
model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

X_train_t = X_train_final[:N_TRAIN_TRANSFER]
y_train_t = y_train_final[:N_TRAIN_TRANSFER]

print("Mulai training Transfer Learning (ResNet50)...")
history_transfer = model_transfer.fit(
    X_train_t, y_train_t,
    epochs=EPOCHS_TRANSFER,
    batch_size=64,
    validation_data=(X_val, y_val),
    verbose=1
)
print("\nTraining selesai.")

val_acc_transfer = history_transfer.history['val_accuracy'][-1]
print(f"Akurasi Transfer Learning pada data validation (epoch terakhir): {val_acc_transfer:.1%}")

### Perbandingan CNN dari Nol vs Transfer Learning — di Data Validation

Sebelum menyentuh data test, kita bandingkan dulu performa kedua model di data validation — data yang tidak dipakai untuk melatih model manapun.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Akurasi validation
models = ['CNN dari Nol', 'Transfer Learning\n(ResNet50)']
val_accs = [val_acc_cnn, val_acc_transfer]
colors = ['#42A5F5', '#66BB6A']
bars = ax1.bar(models, val_accs, color=colors, edgecolor='white', linewidth=2)
for bar, acc in zip(bars, val_accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{acc:.1%}', ha='center', fontweight='bold', fontsize=13)
ax1.set_ylabel('Akurasi Validation')
ax1.set_title('Perbandingan Akurasi Validation', fontweight='bold')
ax1.set_ylim(0, 1)
ax1.grid(axis='y', alpha=0.3)

# Training curves
ax2.plot(history_cnn.history['val_accuracy'], label='CNN dari Nol', linewidth=2, color='#42A5F5')
ax2.plot(history_transfer.history['val_accuracy'], label='Transfer Learning', linewidth=2, color='#66BB6A')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation Accuracy')
ax2.set_title('Kecepatan Belajar', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('CNN vs Transfer Learning — Validation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

pemenang_val = 'CNN dari Nol' if val_acc_cnn > val_acc_transfer else 'Transfer Learning (ResNet50)'
print(f"\nDi data validation, {pemenang_val} lebih unggul "
      f"({max(val_acc_cnn, val_acc_transfer):.1%} vs {min(val_acc_cnn, val_acc_transfer):.1%}).")

### Evaluasi Akhir di Data Test (Sekali, untuk Kedua Model)

Data test belum pernah dilihat oleh proses training maupun proses perbandingan di atas. Karena perbandingan model sudah dilakukan di data validation, data test sekarang dipakai **sekali** untuk kedua model — CNN dari nol dan Transfer Learning — sebagai laporan akhir yang tidak bias oleh proses pemilihan model.

In [ ]:
test_loss, test_acc = model_cnn.evaluate(X_test, y_test, verbose=0)
test_loss_t, test_acc_t = model_transfer.evaluate(X_test, y_test, verbose=0)

print(f"Akurasi CNN dari Nol pada data test      : {test_acc:.1%}")
print(f"Akurasi Transfer Learning pada data test : {test_acc_t:.1%}")

fig, ax = plt.subplots(figsize=(6, 5))
models = ['CNN dari Nol', 'Transfer Learning\n(ResNet50)']
test_accs = [test_acc, test_acc_t]
colors = ['#42A5F5', '#66BB6A']
bars = ax.bar(models, test_accs, color=colors, edgecolor='white', linewidth=2)
for bar, acc in zip(bars, test_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.1%}', ha='center', fontweight='bold', fontsize=13)
ax.set_ylabel('Akurasi Test')
ax.set_title('Perbandingan Akurasi Test (Final)', fontweight='bold')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

if test_acc_t > test_acc:
    print("\nDi data test, Transfer Learning (ResNet50) lebih unggul dibanding CNN dari nol.")
else:
    print("\nDi data test, CNN dari nol tidak kalah dibanding Transfer Learning (ResNet50).")
print("ResNet50 dilatih di ImageNet dengan gambar 224×224; pada gambar CIFAR-10 yang hanya 32×32,")
print("filter-filter yang dipelajari dari ImageNet kurang relevan di resolusi ini. Kalau resolusi gambar")
print("mendekati 224×224 (foto biasa, bukan thumbnail kecil), Transfer Learning cenderung lebih unggul")
print("dibanding CNN kecil yang dilatih dari nol.")

## Bagian 6: Peran NVIDIA dalam CNN & Computer Vision

- **CNN pertama kali populer** berkat GPU NVIDIA — AlexNet (2012) dilatih pada 2 GPU NVIDIA GTX 580
- **cuDNN** mengoptimasi operasi konvolusi di GPU — mempercepat operasi inti CNN secara signifikan dibanding CPU
- **NVIDIA TensorRT** mengoptimasi model CNN untuk deployment (inference lebih cepat; angka pastinya tergantung model dan hardware)
- **NVIDIA Jetson** — perangkat edge computing yang menjalankan CNN untuk autonomous driving, robotics, dll
- **Model ImageNet** yang kita pakai (ResNet50) awalnya dilatih menggunakan GPU NVIDIA

> Fun fact: Training ResNet50 pada ImageNet (14 juta gambar) butuh sekitar satu minggu dengan 8 GPU NVIDIA P100. Tanpa GPU, bisa berbulan-bulan.

In [ ]:
import time

# Benchmark CNN inference
test_batch = X_test[:100]

if tf.config.list_physical_devices('GPU'):
    # GPU inference
    start = time.time()
    for _ in range(10):
        _ = model_cnn.predict(test_batch, verbose=0)
    gpu_time = (time.time() - start) / 10

    print(f"Benchmark CNN Inference (100 gambar):")
    print(f"   GPU: {gpu_time*1000:.1f} ms")
    print(f"\n   GPU NVIDIA mempercepat inference CNN dibanding CPU.")
    print(f"   Ini penting untuk aplikasi real-time: self-driving car, face recognition, dll.")
else:
    start = time.time()
    for _ in range(10):
        _ = model_cnn.predict(test_batch, verbose=0)
    cpu_time = (time.time() - start) / 10

    print(f"Benchmark CNN Inference (100 gambar):")
    print(f"   CPU: {cpu_time*1000:.1f} ms")
    print(f"\nDengan GPU NVIDIA, inference biasanya jauh lebih cepat dibanding CPU")
    print(f"(percepatannya bervariasi tergantung model dan ukuran batch).")
    print(f"Aktifkan GPU di Colab untuk merasakan perbedaannya.")

## 🏋️ Latihan

1. Ganti `RandomRotation(0.04)` menjadi `RandomRotation(0.1)` (rotasi lebih besar), latih ulang CNN dengan `EPOCHS_CNN` yang sama, lalu bandingkan `val_accuracy` epoch terakhir dengan hasil sebelumnya.
2. Coba bekukan hanya separuh layer ResNet50 (`base_model.layers[:N].trainable = False`, sisanya `True`), latih ulang Transfer Learning, lalu bandingkan `val_accuracy`-nya dengan versi backbone yang dibekukan seluruhnya.
3. Tambahkan satu blok Conv2D+BatchNorm+MaxPooling lagi (Block 4, misalnya 256 filter) ke `model_cnn` sebelum `Flatten`, latih ulang, lalu cek apakah `val_accuracy` naik atau malah turun (tanda overfitting).

*Petunjuk:* Simpan angka `val_accuracy` sebelum dan sesudah perubahan supaya perbandingannya jelas.

In [ ]:
# TODO: tulis kodemu di sini

## Kesimpulan

Selamat! Kamu telah menyelesaikan notebook CNN dan Transfer Learning di CIFAR-10. Berikut yang telah kita pelajari:

- Memahami cara kerja CNN: Convolution → Pooling → Classification
- Membangun CNN dari nol untuk klasifikasi gambar CIFAR-10 (10 kelas)
- Augmentasi data lewat layer Keras (`RandomFlip`, `RandomRotation`, `RandomZoom`) di dalam model, bukan generator terpisah
- Memvisualisasikan feature maps — apa yang 'dilihat' CNN
- Menerapkan Transfer Learning dengan ResNet50
- Membandingkan CNN dari nol vs Transfer Learning di data validation, lalu sekali di data test

**Poin penting Transfer Learning:**
- ResNet50 dilatih di ImageNet dengan gambar 224×224. Pada gambar kecil seperti CIFAR-10 (32×32), keunggulan Transfer Learning tidak otomatis terjadi — lihat hasil test di atas.
- Kalau resolusi gambar mendekati 224×224 (foto biasa, bukan thumbnail kecil), Transfer Learning cenderung lebih unggul dibanding CNN kecil yang dilatih dari nol.
- Transfer Learning tetap sangat berguna saat dataset kita kecil (ribuan gambar, bukan jutaan) dan resolusinya wajar.

🔜 **Selanjutnya:** RNN & LSTM untuk data berurutan (teks, time series)